### [GitHub](https://github.com/msultanmahmud/Stress_WESAD/blob/master/Multiclass_Pipeline_Individual_sub_INTEC.ipynb)

In [3]:
import numpy as np
import pandas as pd

In [5]:
# ! pip list

In [7]:
# ! pip install lightgbm

In [9]:
import numpy as np
import pandas as pd
import os
import sys
import random
import warnings
warnings.filterwarnings('ignore')
import pickle
from sklearn.model_selection import train_test_split
from sklearn import svm, metrics,preprocessing
#from sklearn import datasets
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, GridSearchCV,cross_val_score
from sklearn.metrics import accuracy_score,confusion_matrix,ConfusionMatrixDisplay,roc_curve, auc,classification_report
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.model_selection import cross_val_predict
from matplotlib import pyplot as plt
from collections import Counter
from scipy.stats import norm
import seaborn as sns; sns.set(font_scale=1.2)
%matplotlib inline
import time
import xgboost as xgb
import lightgbm as lgb

#### Redaing file

In [11]:
# with open("wsc_visit1_allsub.pkl", "rb") as f:
#     df = pickle.load(f)
#     df=df.sort_values(by='subject')

In [12]:
df=pd.read_csv('dreamt_preprocessed.csv')
df.subject.nunique()

100

In [15]:
print(df.shape)

(95147, 289)


In [18]:
# Identify columns to drop if they exist
columns_to_drop = [col for col in df.columns if 'skewness' in col.lower() or 'kurtosis' in col.lower()]

# Drop only those columns that actually exist
df = df.drop(columns=[col for col in columns_to_drop if col in df.columns])
df 

,subject,epoch,label,C4-M1_min,C4-M1_max,C4-M1_mean,C4-M1_std,C4-M1_p5,C4-M1_p25,C4-M1_p50,...,HR_p95,IBI_min,IBI_max,IBI_mean,IBI_std,IBI_p5,IBI_p25,IBI_p50,IBI_p75,IBI_p95
0,S002,1,3.0,-0.000008,0.000008,6.380306e-09,0.000002,-0.000004,-1.318379e-06,4.882887e-08,...,72.85,1.062500,1.062500,1.062500,0.000000,1.062500,1.062500,1.062500,1.062500,1.062500
1,S002,2,3.0,-0.000006,0.000005,4.531319e-08,0.000002,-0.000003,-1.123064e-06,-4.882887e-08,...,72.85,1.062500,1.062500,1.062500,0.000000,1.062500,1.062500,1.062500,1.062500,1.062500
2,S002,3,3.0,-0.000004,0.000005,4.547595e-08,0.000002,-0.000002,-9.277485e-07,4.882887e-08,...,72.85,1.062500,1.062500,1.062500,0.000000,1.062500,1.062500,1.062500,1.062500,1.062500
3,S002,4,3.0,-0.000017,0.000019,6.071056e-08,0.000003,-0.000004,-1.220722e-06,1.464866e-07,...,72.85,1.062500,1.062500,1.062500,0.000000,1.062500,1.062500,1.062500,1.062500,1.062500
4,S002,5,3.0,-0.000008,0.000009,6.067801e-08,0.000002,-0.000004,-1.416037e-06,4.882887e-08,...,72.85,1.062500,1.062500,1.062500,0.000000,1.062500,1.062500,1.062500,1.062500,1.062500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95142,S103,899,1.0,-0.000025,0.000045,2.679077e-08,0.000008,-0.000012,-4.638743e-06,-1.464866e-07,...,72.12,0.859375,0.906250,0.884599,0.016580,0.859375,0.859375,0.890625,0.890625,0.906250
95143,S103,900,1.0,-0.000035,0.000032,1.027685e-07,0.000008,-0.000014,-4.638743e-06,1.464866e-07,...,68.87,0.812500,0.890625,0.869016,0.019533,0.828125,0.859375,0.859375,0.890625,0.890625
95144,S103,901,1.0,-0.000083,0.000080,1.070654e-07,0.000011,-0.000013,-4.541085e-06,3.906310e-07,...,69.03,0.843750,0.906250,0.871021,0.019181,0.843750,0.859375,0.875000,0.875000,0.906250
95145,S103,902,1.0,-0.000049,0.000056,8.274866e-08,0.000009,-0.000013,-4.931716e-06,1.464866e-07,...,69.42,0.734375,0.937500,0.869594,0.061489,0.734375,0.875000,0.890625,0.906250,0.921875


## Taking the cleaned final data

In [20]:
df = df[~df['subject'].isin(['S062', 'S097'])]
pre_procss_all=df.copy()
pre_procss_all

,subject,epoch,label,C4-M1_min,C4-M1_max,C4-M1_mean,C4-M1_std,C4-M1_p5,C4-M1_p25,C4-M1_p50,...,HR_p95,IBI_min,IBI_max,IBI_mean,IBI_std,IBI_p5,IBI_p25,IBI_p50,IBI_p75,IBI_p95
0,S002,1,3.0,-0.000008,0.000008,6.380306e-09,0.000002,-0.000004,-1.318379e-06,4.882887e-08,...,72.85,1.062500,1.062500,1.062500,0.000000,1.062500,1.062500,1.062500,1.062500,1.062500
1,S002,2,3.0,-0.000006,0.000005,4.531319e-08,0.000002,-0.000003,-1.123064e-06,-4.882887e-08,...,72.85,1.062500,1.062500,1.062500,0.000000,1.062500,1.062500,1.062500,1.062500,1.062500
2,S002,3,3.0,-0.000004,0.000005,4.547595e-08,0.000002,-0.000002,-9.277485e-07,4.882887e-08,...,72.85,1.062500,1.062500,1.062500,0.000000,1.062500,1.062500,1.062500,1.062500,1.062500
3,S002,4,3.0,-0.000017,0.000019,6.071056e-08,0.000003,-0.000004,-1.220722e-06,1.464866e-07,...,72.85,1.062500,1.062500,1.062500,0.000000,1.062500,1.062500,1.062500,1.062500,1.062500
4,S002,5,3.0,-0.000008,0.000009,6.067801e-08,0.000002,-0.000004,-1.416037e-06,4.882887e-08,...,72.85,1.062500,1.062500,1.062500,0.000000,1.062500,1.062500,1.062500,1.062500,1.062500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95142,S103,899,1.0,-0.000025,0.000045,2.679077e-08,0.000008,-0.000012,-4.638743e-06,-1.464866e-07,...,72.12,0.859375,0.906250,0.884599,0.016580,0.859375,0.859375,0.890625,0.890625,0.906250
95143,S103,900,1.0,-0.000035,0.000032,1.027685e-07,0.000008,-0.000014,-4.638743e-06,1.464866e-07,...,68.87,0.812500,0.890625,0.869016,0.019533,0.828125,0.859375,0.859375,0.890625,0.890625
95144,S103,901,1.0,-0.000083,0.000080,1.070654e-07,0.000011,-0.000013,-4.541085e-06,3.906310e-07,...,69.03,0.843750,0.906250,0.871021,0.019181,0.843750,0.859375,0.875000,0.875000,0.906250
95145,S103,902,1.0,-0.000049,0.000056,8.274866e-08,0.000009,-0.000013,-4.931716e-06,1.464866e-07,...,69.42,0.734375,0.937500,0.869594,0.061489,0.734375,0.875000,0.890625,0.906250,0.921875


In [24]:
list(pre_procss_all.columns)

['subject',
 'epoch',
 'label',
 'C4-M1_min',
 'C4-M1_max',
 'C4-M1_mean',
 'C4-M1_std',
 'C4-M1_p5',
 'C4-M1_p25',
 'C4-M1_p50',
 'C4-M1_p75',
 'C4-M1_p95',
 'F4-M1_min',
 'F4-M1_max',
 'F4-M1_mean',
 'F4-M1_std',
 'F4-M1_p5',
 'F4-M1_p25',
 'F4-M1_p50',
 'F4-M1_p75',
 'F4-M1_p95',
 'O2-M1_min',
 'O2-M1_max',
 'O2-M1_mean',
 'O2-M1_std',
 'O2-M1_p5',
 'O2-M1_p25',
 'O2-M1_p50',
 'O2-M1_p75',
 'O2-M1_p95',
 'FP1-O2_min',
 'FP1-O2_max',
 'FP1-O2_mean',
 'FP1-O2_std',
 'FP1-O2_p5',
 'FP1-O2_p25',
 'FP1-O2_p50',
 'FP1-O2_p75',
 'FP1-O2_p95',
 'T3 - CZ_min',
 'T3 - CZ_max',
 'T3 - CZ_mean',
 'T3 - CZ_std',
 'T3 - CZ_p5',
 'T3 - CZ_p25',
 'T3 - CZ_p50',
 'T3 - CZ_p75',
 'T3 - CZ_p95',
 'CZ - T4_min',
 'CZ - T4_max',
 'CZ - T4_mean',
 'CZ - T4_std',
 'CZ - T4_p5',
 'CZ - T4_p25',
 'CZ - T4_p50',
 'CZ - T4_p75',
 'CZ - T4_p95',
 'E1_min',
 'E1_max',
 'E1_mean',
 'E1_std',
 'E1_p5',
 'E1_p25',
 'E1_p50',
 'E1_p75',
 'E1_p95',
 'E2_min',
 'E2_max',
 'E2_mean',
 'E2_std',
 'E2_p5',
 'E2_p25',
 '

In [28]:
# ch=(237-3)/9
26*9

234

## PPG Data

In [34]:
wanted_keywords = ['subject','epoch','BVP', 'EDA', 'TEMP', 'ACC_X', 'ACC_Y', 'ACC_Z', 'HR', 'IBI','label']
pattern = '|'.join(wanted_keywords)  # Creates 'BVP|EDA|TEMP|ACC_X|ACC_Y|ACC_Z|HR|IBI'

# Select matching columns
df_ppg = pre_procss_all.filter(regex=pattern, axis=1)
df_ppg
# Drop columns with any null (NaN) values
df_ppg_cleaned = df_ppg.dropna(axis=1, how='any')

# Optionally: print removed columns
ppg_removed_cols = df_ppg.columns[df_ppg.isnull().any()].tolist()
print("Removed columns with nulls:", ppg_removed_cols)

Removed columns with nulls: []


In [36]:
df_ppg_cleaned

,subject,epoch,label,BVP_min,BVP_max,BVP_mean,BVP_std,BVP_p5,BVP_p25,BVP_p50,...,HR_p95,IBI_min,IBI_max,IBI_mean,IBI_std,IBI_p5,IBI_p25,IBI_p50,IBI_p75,IBI_p95
0,S002,1,3.0,25.61,25.61,25.610000,1.065992e-14,25.6100,25.6100,25.610,...,72.85,1.062500,1.062500,1.062500,0.000000,1.062500,1.062500,1.062500,1.062500,1.062500
1,S002,2,3.0,25.61,25.61,25.610000,1.065992e-14,25.6100,25.6100,25.610,...,72.85,1.062500,1.062500,1.062500,0.000000,1.062500,1.062500,1.062500,1.062500,1.062500
2,S002,3,3.0,25.61,25.61,25.610000,1.065992e-14,25.6100,25.6100,25.610,...,72.85,1.062500,1.062500,1.062500,0.000000,1.062500,1.062500,1.062500,1.062500,1.062500
3,S002,4,3.0,25.61,25.61,25.610000,1.065992e-14,25.6100,25.6100,25.610,...,72.85,1.062500,1.062500,1.062500,0.000000,1.062500,1.062500,1.062500,1.062500,1.062500
4,S002,5,3.0,25.61,25.61,25.610000,1.065992e-14,25.6100,25.6100,25.610,...,72.85,1.062500,1.062500,1.062500,0.000000,1.062500,1.062500,1.062500,1.062500,1.062500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95142,S103,899,1.0,-367.93,256.06,0.962330,1.081959e+02,-181.4965,-74.2100,10.170,...,72.12,0.859375,0.906250,0.884599,0.016580,0.859375,0.859375,0.890625,0.890625,0.906250
95143,S103,900,1.0,-294.16,286.07,-1.207843,1.017870e+02,-175.2740,-78.8400,12.155,...,68.87,0.812500,0.890625,0.869016,0.019533,0.828125,0.859375,0.859375,0.890625,0.890625
95144,S103,901,1.0,-369.02,765.79,0.454700,1.310911e+02,-194.9600,-88.5425,11.370,...,69.03,0.843750,0.906250,0.871021,0.019181,0.843750,0.859375,0.875000,0.875000,0.906250
95145,S103,902,1.0,-663.33,392.27,-1.800103,1.145769e+02,-164.7140,-78.2850,8.610,...,69.42,0.734375,0.937500,0.869594,0.061489,0.734375,0.875000,0.890625,0.906250,0.921875


In [40]:
wanted_keywords = ['BVP', 'EDA', 'TEMP', 'ACC_X', 'ACC_Y', 'ACC_Z', 'HR', 'IBI']
pattern = '|'.join(wanted_keywords)

# Select columns that do NOT match the regex pattern
df_psg = pre_procss_all.loc[:, ~pre_procss_all.columns.str.contains(pattern, case=False, regex=True)]
df_psg

# Drop columns with any null (NaN) values
df_psg_cleaned = df_psg.dropna(axis=1, how='any')
# Optionally: print removed columns
psg_removed_cols = df_psg.columns[df_psg.isnull().any()].tolist()
print("Removed columns with nulls:", psg_removed_cols)
df_psg_cleaned

Removed columns with nulls: []


,subject,epoch,label,C4-M1_min,C4-M1_max,C4-M1_mean,C4-M1_std,C4-M1_p5,C4-M1_p25,C4-M1_p50,...,RAT_p95,SAO2_min,SAO2_max,SAO2_mean,SAO2_std,SAO2_p5,SAO2_p25,SAO2_p50,SAO2_p75,SAO2_p95
0,S002,1,3.0,-0.000008,0.000008,6.380306e-09,0.000002,-0.000004,-1.318379e-06,4.882887e-08,...,0.000034,0.000180,0.000186,0.000183,0.000002,0.000180,0.000181,0.000183,0.000185,0.000186
1,S002,2,3.0,-0.000006,0.000005,4.531319e-08,0.000002,-0.000003,-1.123064e-06,-4.882887e-08,...,0.000035,0.000180,0.000186,0.000183,0.000002,0.000180,0.000181,0.000183,0.000185,0.000186
2,S002,3,3.0,-0.000004,0.000005,4.547595e-08,0.000002,-0.000002,-9.277485e-07,4.882887e-08,...,0.000035,0.000180,0.000186,0.000183,0.000002,0.000180,0.000181,0.000183,0.000185,0.000186
3,S002,4,3.0,-0.000017,0.000019,6.071056e-08,0.000003,-0.000004,-1.220722e-06,1.464866e-07,...,0.000035,0.000180,0.000186,0.000183,0.000002,0.000180,0.000181,0.000183,0.000185,0.000186
4,S002,5,3.0,-0.000008,0.000009,6.067801e-08,0.000002,-0.000004,-1.416037e-06,4.882887e-08,...,0.000034,0.000180,0.000186,0.000183,0.000002,0.000180,0.000181,0.000183,0.000185,0.000186
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95142,S103,899,1.0,-0.000025,0.000045,2.679077e-08,0.000008,-0.000012,-4.638743e-06,-1.464866e-07,...,0.000002,0.888365,0.931008,0.914902,0.014043,0.889694,0.900106,0.919753,0.929534,0.929682
95143,S103,900,1.0,-0.000035,0.000032,1.027685e-07,0.000008,-0.000014,-4.638743e-06,1.464866e-07,...,0.000004,0.918381,0.941296,0.926600,0.006995,0.919724,0.919753,0.929598,0.929651,0.939926
95144,S103,901,1.0,-0.000083,0.000080,1.070654e-07,0.000011,-0.000013,-4.541085e-06,3.906310e-07,...,0.000002,0.918450,0.941275,0.929748,0.005847,0.919747,0.929610,0.929641,0.929672,0.939907
95145,S103,902,1.0,-0.000049,0.000056,8.274866e-08,0.000009,-0.000013,-4.931716e-06,1.464866e-07,...,0.000001,0.908547,0.930959,0.917330,0.005934,0.909848,0.909879,0.919720,0.919785,0.929641


In [42]:
df_psg.isnull().sum()
null_counts = df_psg.isnull().sum()
null_counts = null_counts[null_counts > 0]
null_counts

Series([], dtype: int64)

In [44]:
df_ppg.label.value_counts()

label
1.0    44727
3.0    23139
0.0    17490
2.0     7938
Name: count, dtype: int64

In [47]:
# label_mapping = {
#     0: 'Wake Up',
#     1: 'Non-REM',
#     2: 'Non-REM',
#     3: 'Non-REM',
#     4: 'Non-REM',
#     7: 'REM'
# }
###0: wake up;1:NOn-REM;2: REM< 
# df['map_label']=df['label'].map(label_mapping)
# df
# label_mapping = {0: 0,1: 1,2: 1,3: 1,4: 1,5:2,6:3,7: 3}
# df['map_label'] = df['label'].map(label_mapping)

df_ppg_cleaned = df_ppg_cleaned[df_ppg_cleaned['label'].isin([0,1,2])]
df_ppg_cleaned.head(3)
### How many subjec
# sample=df.subject.unique()[:5]
# # # Sample the data (if needed)
# df_sample = df[df.subject.isin(sample)]
# df_sample

,subject,epoch,label,BVP_min,BVP_max,BVP_mean,BVP_std,BVP_p5,BVP_p25,BVP_p50,...,HR_p95,IBI_min,IBI_max,IBI_mean,IBI_std,IBI_p5,IBI_p25,IBI_p50,IBI_p75,IBI_p95
305,S002,307,0.0,-1257.38,705.59,0.654393,172.768194,-327.953,-16.4275,4.025,...,81.62,0.937500,0.953125,0.952734,0.002440,0.953125,0.953125,0.953125,0.953125,0.953125
306,S002,308,0.0,-749.06,472.29,-0.381060,101.202434,-95.095,-22.2800,1.430,...,85.00,0.953125,0.953125,0.953125,0.000000,0.953125,0.953125,0.953125,0.953125,0.953125
307,S002,309,0.0,-281.51,153.36,-1.509670,58.567645,-114.200,-24.3425,2.820,...,82.45,0.812500,0.953125,0.823187,0.037272,0.812500,0.812500,0.812500,0.812500,0.953125


In [49]:
df=df_ppg_cleaned.copy()

In [51]:
df1=df_ppg_cleaned.head(1000)
df1.head(3)

,subject,epoch,label,BVP_min,BVP_max,BVP_mean,BVP_std,BVP_p5,BVP_p25,BVP_p50,...,HR_p95,IBI_min,IBI_max,IBI_mean,IBI_std,IBI_p5,IBI_p25,IBI_p50,IBI_p75,IBI_p95
305,S002,307,0.0,-1257.38,705.59,0.654393,172.768194,-327.953,-16.4275,4.025,...,81.62,0.937500,0.953125,0.952734,0.002440,0.953125,0.953125,0.953125,0.953125,0.953125
306,S002,308,0.0,-749.06,472.29,-0.381060,101.202434,-95.095,-22.2800,1.430,...,85.00,0.953125,0.953125,0.953125,0.000000,0.953125,0.953125,0.953125,0.953125,0.953125
307,S002,309,0.0,-281.51,153.36,-1.509670,58.567645,-114.200,-24.3425,2.820,...,82.45,0.812500,0.953125,0.823187,0.037272,0.812500,0.812500,0.812500,0.812500,0.953125


In [53]:
# df1=df_sample.copy()

In [55]:
# !jupyter kernelspec list

In [57]:
from joblib import Parallel, delayed
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import xgboost as xgb
import lightgbm as lgb
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn import svm
import pandas as pd
import numpy as np

def preprocess_scaling(df):
    '''Scales features and splits into train/test.'''
    X = df.drop(columns=['subject', 'epoch', 'label'])
    y = df['label']
    X_scaled = preprocessing.scale(X)
    return train_test_split(X_scaled, y, test_size=0.2, random_state=42)

def train_and_evaluate(model, X_train, y_train, X_test, y_test, model_name, auc_mode='ovr'):
    try:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
        recall = recall_score(y_test, y_pred, average='macro', zero_division=0)
        f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)

        # AUC calculation — multi-class
        if hasattr(model, "predict_proba"):
            y_probs = model.predict_proba(X_test)
            auc = roc_auc_score(y_test, y_probs, multi_class=auc_mode, average='macro')
        else:
            auc = None  # SVM without probability=True will not support this

        return (model_name, acc, f1, precision, recall, auc)
    except Exception as e:
        print(f"Error with model {model_name}: {e}")
        return (model_name, None, None, None, None, None)

def multiple_model(df, auc_mode='ovr'):
    X_train, X_test, y_train, y_test = preprocess_scaling(df)
    model_name_list = ['KNN', 'DT', 'XGBoost', 'LightGBM', 'RF', 'SVM']

    # SVM needs probability=True for predict_proba
    models = [
        KNeighborsClassifier(metric='minkowski', n_neighbors=2, weights='distance', p=2),
        DecisionTreeClassifier(criterion='entropy', splitter='best', max_depth=50),
        xgb.XGBClassifier(objective='multi:softprob', num_class=len(np.unique(y_train)), random_state=42,
                          max_depth=15, learning_rate=0.3, gamma=0.2, colsample_bytree=0.7,
                          min_child_weight=1, eval_metric='mlogloss', n_jobs=-1),
        lgb.LGBMClassifier(colsample_bytree=0.7, learning_rate=0.3, max_depth=100,
                           min_child_weight=0.5, n_estimators=200, verbose=-1, n_jobs=-1),
        RandomForestClassifier(max_depth=40, max_features='log2', n_estimators=200, n_jobs=-1),
        svm.SVC(C=10, kernel='rbf', degree=3, gamma=0.1, probability=True)  # Required for AUC
    ]

    results_list = Parallel(n_jobs=-1)(delayed(train_and_evaluate)(
        model, X_train, y_train, X_test, y_test, name, auc_mode
    ) for model, name in zip(models, model_name_list))

    results_df = pd.DataFrame(results_list, columns=['Model', 'Accuracy', 'F1', 'Precision', 'Recall', 'AUC'])
    results_df.set_index('Model', inplace=True)
    results_df=results_df[['Accuracy','AUC','Precision', 'Recall','F1']]
    return results_df

In [59]:
# # Default AUC = 'ovr'
# results = multiple_model(df1)
# results

In [61]:
# # Or try one-vs-one AUC if OvR seems low
# results_ovo = multiple_model(df1, auc_mode='ovo')
# results_ovo

### Ensemble Learning

In [65]:
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import numpy as np
import pandas as pd
import xgboost as xgb
import lightgbm as lgb
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn import svm

def ensemble_model(df, voting_type='hard', auc_mode='ovr'):
    X_train, X_test, y_train, y_test = preprocess_scaling(df)

    # Define base models
    knn = KNeighborsClassifier(metric='minkowski', n_neighbors=2, weights='distance', p=2)
    dt = DecisionTreeClassifier(criterion='entropy', splitter='best', max_depth=50)
    xgb_model = xgb.XGBClassifier(objective='multi:softprob', random_state=42, min_child_weight=1,
                                  max_depth=15, learning_rate=0.3, gamma=0.2, colsample_bytree=0.7,
                                  eval_metric='mlogloss', n_jobs=-1, use_label_encoder=False)
    lgb_model = lgb.LGBMClassifier(colsample_bytree=0.7, learning_rate=0.3, max_depth=100,
                                   min_child_weight=0.5, n_estimators=200, verbose=-1, n_jobs=-1)
    rf = RandomForestClassifier(max_depth=40, max_features='log2', n_estimators=200, n_jobs=-1)
    svc = svm.SVC(C=10, kernel='rbf', degree=3, gamma=0.1, probability=(voting_type == 'soft'))

    # Create ensemble using voting
    ensemble = VotingClassifier(
        estimators=[
            ('KNN', knn),
            ('DT', dt),
            ('XGB', xgb_model),
            ('LGBM', lgb_model),
            ('RF', rf),
            ('SVM', svc)
        ],
        voting=voting_type,
        n_jobs=-1
    )

    ensemble.fit(X_train, y_train)
    y_pred = ensemble.predict(X_test)

    # Compute metrics
    acc = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
    recall = recall_score(y_test, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)

    # Compute AUC only if voting_type is 'soft'
    try:
        if voting_type == 'soft':
            y_probs = ensemble.predict_proba(X_test)
            auc = roc_auc_score(y_test, y_probs, multi_class=auc_mode, average='macro')
        else:
            auc = None
    except Exception as e:
        print(f"AUC error: {e}")
        auc = None

    results = pd.DataFrame([{
        'Model': f'VotingClassifier ({voting_type})',
        'Accuracy': acc,
        'AUC': auc,
        'Precision': precision,
        'Recall': recall,
        'F1': f1

    }]).set_index('Model')

    return results

## Classification using all faetures
- calling the function

In [68]:
df1

,subject,epoch,label,BVP_min,BVP_max,BVP_mean,BVP_std,BVP_p5,BVP_p25,BVP_p50,...,HR_p95,IBI_min,IBI_max,IBI_mean,IBI_std,IBI_p5,IBI_p25,IBI_p50,IBI_p75,IBI_p95
305,S002,307,0.0,-1257.38,705.59,0.654393,172.768194,-327.9530,-16.4275,4.025,...,81.62,0.937500,0.953125,0.952734,0.002440,0.953125,0.953125,0.953125,0.953125,0.953125
306,S002,308,0.0,-749.06,472.29,-0.381060,101.202434,-95.0950,-22.2800,1.430,...,85.00,0.953125,0.953125,0.953125,0.000000,0.953125,0.953125,0.953125,0.953125,0.953125
307,S002,309,0.0,-281.51,153.36,-1.509670,58.567645,-114.2000,-24.3425,2.820,...,82.45,0.812500,0.953125,0.823187,0.037272,0.812500,0.812500,0.812500,0.812500,0.953125
308,S002,310,0.0,-369.24,242.02,1.410377,74.185047,-126.8245,-37.6350,6.530,...,85.88,0.812500,0.812500,0.812500,0.000000,0.812500,0.812500,0.812500,0.812500,0.812500
309,S002,311,0.0,-572.48,620.29,-0.499120,154.295774,-286.6220,-42.6350,1.215,...,87.73,0.765625,0.968750,0.781552,0.046296,0.765625,0.765625,0.765625,0.765625,0.953125
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1539,S003,735,1.0,-28.20,22.02,-0.042670,11.252605,-19.7105,-9.1800,2.360,...,62.67,0.828125,1.078125,0.947766,0.074704,0.828125,0.890625,0.937500,1.015625,1.062500
1540,S003,736,1.0,-31.32,25.04,-0.028470,11.998739,-20.7500,-9.5775,2.220,...,63.37,0.828125,1.078125,0.944448,0.075328,0.843750,0.859375,0.937500,1.000000,1.062500
1541,S003,737,1.0,-29.80,23.49,0.059117,12.348117,-21.3610,-10.0375,2.530,...,63.70,0.828125,1.062500,0.931281,0.066094,0.828125,0.875000,0.937500,0.984375,1.046875
1542,S003,738,1.0,-31.10,22.66,-0.017603,12.335570,-21.1300,-9.9525,2.240,...,64.05,0.812500,1.078125,0.952885,0.068992,0.843750,0.890625,0.968750,1.000000,1.046875


### PPG

In [73]:
# Default AUC = 'ovr'
cls_reg_ppg = multiple_model(df_ppg_cleaned)
cls_reg_ppg

,Accuracy,AUC,Precision,Recall,F1
Model,,,,,
KNN,0.864942,0.913786,0.832048,0.830300,0.830559
DT,0.798945,0.807910,0.750986,0.748083,0.749438
XGBoost,0.902074,0.971985,0.907144,0.861982,0.882647
LightGBM,0.904640,0.971638,0.907038,0.869300,0.886726
RF,0.913976,0.977943,0.918147,0.882677,0.899162
SVM,0.877771,0.950880,0.860035,0.841205,0.850247


In [74]:
# Or try one-vs-one AUC if OvR seems low
cls_reg_ovo_ppg = multiple_model(df_ppg_cleaned, auc_mode='ovo')
cls_reg_ovo_ppg

,Accuracy,AUC,Precision,Recall,F1
Model,,,,,
KNN,0.864942,0.916945,0.832048,0.830300,0.830559
DT,0.801440,0.810359,0.754469,0.747145,0.750660
XGBoost,0.902074,0.972915,0.907144,0.861982,0.882647
LightGBM,0.904640,0.974301,0.907038,0.869300,0.886726
RF,0.913976,0.979672,0.917644,0.883264,0.899314
SVM,0.877771,0.954289,0.860035,0.841205,0.850247


### PSG 

In [76]:
# Default AUC = 'ovr'
cls_reg_psg = multiple_model(df_psg_cleaned)
cls_reg_psg

,Accuracy,AUC,Precision,Recall,F1
Model,,,,,
KNN,0.906962,0.949873,0.888586,0.873865,0.880848
DT,0.863605,0.886117,0.823700,0.822581,0.823135
XGBoost,0.935634,0.992944,0.928430,0.910523,0.919047
LightGBM,0.948175,0.994344,0.941455,0.931320,0.936246
RF,0.924648,0.989502,0.920741,0.885113,0.901014
SVM,0.900584,0.984331,0.892555,0.875375,0.879637


In [ ]:
# Or try one-vs-one AUC if OvR seems low
cls_reg_ovo_psg = multiple_model(df_psg_cleaned, auc_mode='ovo')
cls_reg_ovo_psg

### Calling Ensemble

In [ ]:
# res_ens=ensemble_model(df1)
# res_ens

In [ ]:
# # Hard voting — no AUC
# results_hard = ensemble_model(df1, voting_type='hard')
# results_hard

In [78]:
# Soft voting — AUC with OvR
results_soft = ensemble_model(df1, voting_type='soft', auc_mode='ovr')
results_soft

,Accuracy,AUC,Precision,Recall,F1
Model,,,,,
VotingClassifier (soft),0.965,0.985335,0.951092,0.896667,0.920792


In [79]:
# Or try OvO instead
results_soft_ovo = ensemble_model(df1, voting_type='soft', auc_mode='ovo')
results_soft_ovo

,Accuracy,AUC,Precision,Recall,F1
Model,,,,,
VotingClassifier (soft),0.96,0.975389,0.946626,0.875833,0.905631
